# Preparação dos trechos para busca semântica

Vamos dividir os textos do corpus em trechos, preservando suas
referências e metadados. Depois utilizaremos esses trechos
no experimento de recuperação com Transformer.

In [1]:
from pathlib import Path
import csv

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

caminho_corpus = raiz / "data" / "processed" / "corpus_v3.csv"

with caminho_corpus.open(
    encoding="utf-8-sig", newline=""
) as arquivo:
    corpus = list(csv.DictReader(arquivo, delimiter=";"))

sem_texto = [
    registro for registro in corpus
    if not registro["texto_limpo"].strip()
]

print("Registros carregados:", len(corpus))
print("Fontes distintas:", len({r["arquivo"] for r in corpus}))
print("Registros sem texto limpo:", len(sem_texto))

Registros carregados: 89
Fontes distintas: 9
Registros sem texto limpo: 0


In [2]:
%pip install sentence-transformers

  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (4.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 1.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 855.2/855.2 kB 2.1 MB/s  0:00:00 eta 0:00:01
Using cached click-8.5.0-py3-none-any.whl (125 kB)
Using cached hf_xet-1.7.0-cp38-abi3-macosx_11_0_arm64.whl (3.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 6.2 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 7.2 MB/s  0:00:00 eta 0:00:01
Using cached idna-3.20-py3-none-any.whl (69 kB)
Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl (484 kB)
  Attempting uninstall: regex
    Found existing installation: regex 2025.9.1
    Uninstalling regex-2025.9.1:
      Successful

In [3]:
from sentence_transformers import SentenceTransformer

nome_modelo = (
    "PORTULAN/serafim-100m-portuguese-pt-sentence-encoder-ir"
)

modelo_embeddings = SentenceTransformer(nome_modelo)
tokenizador = modelo_embeddings.tokenizer

print("Modelo:", nome_modelo)
print("Limite de tokens:", modelo_embeddings.max_seq_length)
print(
    "Dimensão dos embeddings:",
    modelo_embeddings.get_sentence_embedding_dimension(),
)

W1027 14:40:43.683000 82719 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.68k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/678k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Modelo: PORTULAN/serafim-100m-portuguese-pt-sentence-encoder-ir
Limite de tokens: 128
Dimensão dos embeddings: 768


/var/folders/62/8psz309j2577h4d2_wz1fd100000gn/T/ipykernel_82719/1214989654.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  modelo_embeddings.get_sentence_embedding_dimension(),


In [4]:
limite_tokens = modelo_embeddings.max_seq_length
tokens_especiais = tokenizador.num_special_tokens_to_add(pair=False)

tamanhos_tokens = [
    len(tokenizador.encode(
        registro["texto_limpo"],
        add_special_tokens=False,
        truncation=False,
        verbose=False,
    ))
    for registro in corpus
]

limite_conteudo = limite_tokens - tokens_especiais

print("Limite total:", limite_tokens)
print("Tokens especiais:", tokens_especiais)
print("Espaço para conteúdo:", limite_conteudo)
print("Menor texto:", min(tamanhos_tokens), "tokens")
print("Maior texto:", max(tamanhos_tokens), "tokens")
print(
    "Registros que precisam ser divididos:",
    sum(tamanho > limite_conteudo for tamanho in tamanhos_tokens),
)

Limite total: 128
Tokens especiais: 2
Espaço para conteúdo: 126
Menor texto: 18 tokens
Maior texto: 1348 tokens
Registros que precisam ser divididos: 74


In [5]:
def dividir_em_trechos(texto, tamanho=120, sobreposicao=20):
    assert 0 <= sobreposicao < tamanho <= limite_conteudo
    assert tokenizador.is_fast, "Precisamos de um tokenizador rápido."

    codificacao = tokenizador(
        texto,
        add_special_tokens=False,
        truncation=False,
        return_offsets_mapping=True,
        verbose=False,
    )

    posicoes = codificacao["offset_mapping"]
    trechos = []
    inicio = 0

    while inicio < len(posicoes):
        fim = min(inicio + tamanho, len(posicoes))

        # Confere novamente o trecho isolado, pois a tokenização
        # pode mudar nos limites do corte.
        while True:
            inicio_char = posicoes[inicio][0]
            fim_char = posicoes[fim - 1][1]
            texto_trecho = texto[inicio_char:fim_char]

            quantidade = len(tokenizador.encode(
                texto_trecho,
                add_special_tokens=False,
                truncation=False,
                verbose=False,
            ))

            if quantidade <= tamanho:
                break

            fim -= 1
            assert fim > inicio, "Não foi possível criar o trecho."

        trechos.append({
            "texto_trecho": texto_trecho,
            "inicio_char": inicio_char,
            "fim_char": fim_char,
            "tokens_conteudo": quantidade,
        })

        if fim == len(posicoes):
            break

        inicio = max(inicio + 1, fim - sobreposicao)

    return trechos


indice_maior = max(
    range(len(corpus)),
    key=lambda indice: tamanhos_tokens[indice],
)

amostra_trechos = dividir_em_trechos(
    corpus[indice_maior]["texto_limpo"]
)

print("Arquivo:", corpus[indice_maior]["arquivo"])
print("Tokens do registro:", tamanhos_tokens[indice_maior])
print("Trechos gerados:", len(amostra_trechos))
print("Tokens por trecho:", [
    trecho["tokens_conteudo"] for trecho in amostra_trechos
])
print("\nPrimeiro trecho:\n")
print(amostra_trechos[0]["texto_trecho"])

Arquivo: 05.html
Tokens do registro: 1348
Trechos gerados: 14
Tokens por trecho: [120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 48]

Primeiro trecho:

Introdução

Acabou de ingressar em uma graduação do Ibmec? Seja bem-vindo calouro!

Neste Guia do Estudante você irá encontrar tudo que precisa para iniciar sua jornada acadêmica da melhor forma, com links úteis, detalhes sobre acessos, passo a passo para fazer sua carteirinha de estudante e muito mais!

Sou calouro Ibmec, e agora?

Antes de mais nada, parabéns, agora você faz parte de uma das principais instituições de ensino superior do país.

Para acessar os ambientes online do Ibmec e as ferramentas disponibilizadas aos nossos alunos


In [6]:
trechos_corpus = []

for numero_registro, registro in enumerate(corpus, start=1):
    partes = dividir_em_trechos(registro["texto_limpo"])

    for numero_trecho, parte in enumerate(partes, start=1):
        trechos_corpus.append({
            **registro,
            "id_registro": f"r{numero_registro:03d}",
            "id_trecho": f"r{numero_registro:03d}_t{numero_trecho:03d}",
            "numero_trecho": numero_trecho,
            **parte,
        })

assert len({
    trecho["id_trecho"] for trecho in trechos_corpus
}) == len(trechos_corpus)

assert all(
    trecho["texto_trecho"].strip()
    for trecho in trechos_corpus
)

# Confere a entrada completa, incluindo os tokens especiais.
tamanhos_entrada = [
    len(tokenizador.encode(
        trecho["texto_trecho"],
        add_special_tokens=True,
        truncation=False,
        verbose=False,
    ))
    for trecho in trechos_corpus
]

assert max(tamanhos_entrada) <= limite_tokens

print("Registros representados:", len({
    trecho["id_registro"] for trecho in trechos_corpus
}))
print("Trechos gerados:", len(trechos_corpus))
print("Fontes preservadas:", len({
    trecho["arquivo"] for trecho in trechos_corpus
}))
print("Maior entrada com tokens especiais:", max(tamanhos_entrada))

Registros representados: 89
Trechos gerados: 279
Fontes preservadas: 9
Maior entrada com tokens especiais: 122


In [7]:
for numero_registro, registro in enumerate(corpus, start=1):
    id_registro = f"r{numero_registro:03d}"
    texto_original = registro["texto_limpo"]

    partes = [
        trecho for trecho in trechos_corpus
        if trecho["id_registro"] == id_registro
    ]

    cobertura = [False] * len(texto_original)

    for parte in partes:
        inicio = parte["inicio_char"]
        fim = parte["fim_char"]

        assert texto_original[inicio:fim] == parte["texto_trecho"]

        for posicao in range(inicio, fim):
            cobertura[posicao] = True

    caracteres_sem_cobertura = [
        posicao
        for posicao, caractere in enumerate(texto_original)
        if not caractere.isspace() and not cobertura[posicao]
    ]

    assert not caracteres_sem_cobertura, (
        f"Conteúdo sem cobertura em {id_registro}"
    )

print("89 registros conferidos.")
print("Recortes correspondem aos textos de origem.")
print("Todos os caracteres não brancos estão cobertos.")

89 registros conferidos.
Recortes correspondem aos textos de origem.
Todos os caracteres não brancos estão cobertos.


In [8]:
pasta_saida = raiz / "data" / "processed"

arquivos_trechos = {
    "trechos_v1.csv": trechos_corpus,
    "configuracao_trechos_v1.csv": [{
        "corpus_origem": "corpus_v3.csv",
        "modelo": nome_modelo,
        "estrategia": "Janelas de tokens com sobreposição",
        "tamanho_conteudo": 120,
        "sobreposicao_tokens_origem": 20,
        "limite_modelo": limite_tokens,
        "tokens_especiais": tokens_especiais,
        "registros_origem": len(corpus),
        "quantidade_trechos": len(trechos_corpus),
    }],
}

# Confere todos os destinos antes de iniciar a gravação.
for nome in arquivos_trechos:
    if (pasta_saida / nome).exists():
        raise FileExistsError(f"{nome} já existe.")

for nome, linhas in arquivos_trechos.items():
    caminho = pasta_saida / nome
    colunas = list(linhas[0])

    with caminho.open("x", encoding="utf-8-sig", newline="") as arquivo:
        escritor = csv.DictWriter(
            arquivo,
            fieldnames=colunas,
            delimiter=";",
        )
        escritor.writeheader()
        escritor.writerows(linhas)

    with caminho.open(encoding="utf-8-sig", newline="") as arquivo:
        recuperadas = list(csv.DictReader(arquivo, delimiter=";"))

    esperadas = [
        {chave: str(valor) for chave, valor in linha.items()}
        for linha in linhas
    ]

    assert recuperadas == esperadas, f"Falha na gravação de {nome}"
    print(f"{nome}: {len(recuperadas)} registros, conferido")

trechos_v1.csv: 279 registros, conferido
configuracao_trechos_v1.csv: 1 registros, conferido


## Geração de embeddings com Transformer

Utilizaremos o Serafim para representar cada trecho como um vetor
de 768 dimensões. Esses vetores serão usados na busca por similaridade.

Normalizaremos os embeddings para que o produto escalar entre
os vetores corresponda à similaridade de cosseno.

In [9]:
import numpy as np

textos_trechos = [
    trecho["texto_trecho"]
    for trecho in trechos_corpus
]

embeddings_trechos = modelo_embeddings.encode(
    textos_trechos,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

assert embeddings_trechos.shape == (
    len(trechos_corpus),
    modelo_embeddings.get_embedding_dimension(),
)
assert np.isfinite(embeddings_trechos).all()

normas = np.linalg.norm(embeddings_trechos, axis=1)
assert np.allclose(normas, 1.0, atol=1e-5)

print("Formato da matriz:", embeddings_trechos.shape)
print("Valores finitos: conferido")
print("Vetores normalizados: conferido")

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Formato da matriz: (279, 768)
Valores finitos: conferido
Vetores normalizados: conferido


In [10]:
pergunta_busca = "Como solicitar o trancamento de matrícula?"

embedding_pergunta = modelo_embeddings.encode(
    pergunta_busca,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

similaridades = embeddings_trechos @ embedding_pergunta
indices_encontrados = np.argsort(-similaridades)[:3]

print("Pergunta:", pergunta_busca)

for posicao, indice in enumerate(indices_encontrados, start=1):
    trecho_encontrado = trechos_corpus[int(indice)]

    print(f"\nResultado {posicao}")
    print("Similaridade:", round(float(similaridades[indice]), 4))
    print("Arquivo:", trecho_encontrado["arquivo"])
    print("Página:", trecho_encontrado.get("pagina") or "Não se aplica")
    print("Mês:", trecho_encontrado.get("mes") or "Não se aplica")
    print("ID:", trecho_encontrado["id_trecho"])
    print("Texto:")
    print(trecho_encontrado["texto_trecho"])

Pergunta: Como solicitar o trancamento de matrícula?

Resultado 1
Similaridade: 0.7972
Arquivo: GradManualAlunoIbmec.pdf
Página: 38
Mês: Não se aplica
ID: r028_t002
Texto:
rícula/Transferência > Entrevista Trancamento/
Cancelamento de Matrícula”, após a unidade entrará em contato para realizar a
entrevista e abrir o requerimento de trancamento de matrícula;
(b) a matrícula deve estar efetiva e vigente ao tempo da solicitação de trancamento;
(c) o aluno deve atender todas as regras relativas ao trancamento da matrícula
previstas no contrato de prestação de serviço, bem como respeitar as condições
estabelecidas na legislação aplicável e os prazos estabelecidos no calendário
acadêmico;
(d) o

Resultado 2
Similaridade: 0.7937
Arquivo: GradManualAlunoIbmec.pdf
Página: 38
Mês: Não se aplica
ID: r028_t001
Texto:
Trancamento de matrícula
Até a data limite constante no calendário acadêmico o aluno poderá solicitar
o trancamento da matrícula. O atendimento da solicitação de trancamento da
matríc

In [11]:
textos_por_registro = {
    f"r{numero:03d}": registro["texto_limpo"]
    for numero, registro in enumerate(corpus, start=1)
}

cortes_em_palavras = []

for trecho in trechos_corpus:
    texto_origem = textos_por_registro[trecho["id_registro"]]
    inicio = trecho["inicio_char"]
    fim = trecho["fim_char"]

    inicio_cortado = (
        inicio > 0
        and texto_origem[inicio - 1].isalnum()
        and texto_origem[inicio].isalnum()
    )

    fim_cortado = (
        fim < len(texto_origem)
        and texto_origem[fim - 1].isalnum()
        and texto_origem[fim].isalnum()
    )

    if inicio_cortado or fim_cortado:
        cortes_em_palavras.append(trecho["id_trecho"])

print("Trechos com possível corte dentro de palavra:",
      len(cortes_em_palavras))
print("Primeiros IDs:", cortes_em_palavras[:10])

Trechos com possível corte dentro de palavra: 60
Primeiros IDs: ['r006_t002', 'r006_t003', 'r010_t003', 'r011_t002', 'r012_t001', 'r013_t005', 'r016_t003', 'r018_t002', 'r019_t002', 'r023_t002']


In [12]:
def dividir_em_trechos_v2(texto, tamanho=120, sobreposicao=20):
    assert 0 <= sobreposicao < tamanho <= limite_conteudo

    codificacao = tokenizador(
        texto,
        add_special_tokens=False,
        truncation=False,
        return_offsets_mapping=True,
        verbose=False,
    )

    posicoes = codificacao["offset_mapping"]

    def corta_palavra(posicao):
        return (
            0 < posicao < len(texto)
            and texto[posicao - 1].isalnum()
            and texto[posicao].isalnum()
        )

    partes = []
    inicio = 0

    while inicio < len(posicoes):
        fim = min(inicio + tamanho, len(posicoes))
        inicio_char = posicoes[inicio][0]

        while fim > inicio:
            fim_char = posicoes[fim - 1][1]

            if corta_palavra(fim_char):
                fim -= 1
                continue

            texto_trecho = texto[inicio_char:fim_char]

            quantidade = len(tokenizador.encode(
                texto_trecho,
                add_special_tokens=False,
                truncation=False,
                verbose=False,
            ))

            if quantidade <= tamanho:
                break

            fim -= 1
        else:
            raise ValueError(
                "Não foi possível formar um trecho com palavra completa."
            )

        assert not corta_palavra(inicio_char)
        assert not corta_palavra(fim_char)

        partes.append({
            "texto_trecho": texto_trecho,
            "inicio_char": inicio_char,
            "fim_char": fim_char,
            "tokens_conteudo": quantidade,
        })

        if fim == len(posicoes):
            break

        proximo_inicio = max(inicio + 1, fim - sobreposicao)

        # Avança até o início de uma palavra completa.
        while (
            proximo_inicio < fim
            and corta_palavra(posicoes[proximo_inicio][0])
        ):
            proximo_inicio += 1

        inicio = proximo_inicio

    return partes


trechos_corpus_v2 = []

for numero, registro in enumerate(corpus, start=1):
    partes = dividir_em_trechos_v2(registro["texto_limpo"])

    for ordem, parte in enumerate(partes, start=1):
        trechos_corpus_v2.append({
            **registro,
            "id_registro": f"r{numero:03d}",
            "id_trecho": f"r{numero:03d}_t{ordem:03d}",
            "numero_trecho": ordem,
            **parte,
        })

print("Trechos v2:", len(trechos_corpus_v2))
print("Limites de palavras: conferidos")
print("Maior conteúdo:", max(
    trecho["tokens_conteudo"] for trecho in trechos_corpus_v2
))

Trechos v2: 279
Limites de palavras: conferidos
Maior conteúdo: 120


In [13]:
for numero, registro in enumerate(corpus, start=1):
    id_registro = f"r{numero:03d}"
    texto_origem = registro["texto_limpo"]

    partes = [
        trecho for trecho in trechos_corpus_v2
        if trecho["id_registro"] == id_registro
    ]

    assert partes, f"Registro sem trechos: {id_registro}"
    cobertura = [False] * len(texto_origem)

    for parte in partes:
        inicio = parte["inicio_char"]
        fim = parte["fim_char"]

        assert texto_origem[inicio:fim] == parte["texto_trecho"]

        for campo, valor in registro.items():
            assert parte[campo] == valor, (
                f"Metadado alterado: {id_registro}, {campo}"
            )

        for posicao in range(inicio, fim):
            cobertura[posicao] = True

    assert all(
        coberto or caractere.isspace()
        for caractere, coberto in zip(texto_origem, cobertura)
    ), f"Conteúdo sem cobertura: {id_registro}"

tamanhos_entrada_v2 = [
    len(tokenizador.encode(
        trecho["texto_trecho"],
        add_special_tokens=True,
        truncation=False,
        verbose=False,
    ))
    for trecho in trechos_corpus_v2
]

assert max(tamanhos_entrada_v2) <= limite_tokens
assert len({t["id_trecho"] for t in trechos_corpus_v2}) == len(
    trechos_corpus_v2
)

print("Registros com cobertura e metadados conferidos:", len(corpus))
print("IDs únicos:", len(trechos_corpus_v2))
print("Maior entrada com tokens especiais:", max(tamanhos_entrada_v2))

Registros com cobertura e metadados conferidos: 89
IDs únicos: 279
Maior entrada com tokens especiais: 122


In [14]:
arquivos_trechos_v2 = {
    "trechos_v2.csv": trechos_corpus_v2,
    "configuracao_trechos_v2.csv": [{
        "corpus_origem": "corpus_v3.csv",
        "modelo": nome_modelo,
        "estrategia": "Janelas de tokens com limites de palavras",
        "tamanho_maximo_conteudo": 120,
        "sobreposicao_alvo_tokens": 20,
        "ajuste_sobreposicao": "Pode diminuir para preservar palavras",
        "limite_modelo": limite_tokens,
        "tokens_especiais": tokens_especiais,
        "registros_origem": len(corpus),
        "quantidade_trechos": len(trechos_corpus_v2),
    }],
}

for nome in arquivos_trechos_v2:
    if (pasta_saida / nome).exists():
        raise FileExistsError(f"{nome} já existe.")

for nome, linhas in arquivos_trechos_v2.items():
    caminho = pasta_saida / nome

    with caminho.open("x", encoding="utf-8-sig", newline="") as arquivo:
        escritor = csv.DictWriter(
            arquivo,
            fieldnames=list(linhas[0]),
            delimiter=";",
        )
        escritor.writeheader()
        escritor.writerows(linhas)

    with caminho.open(encoding="utf-8-sig", newline="") as arquivo:
        recuperadas = list(csv.DictReader(arquivo, delimiter=";"))

    esperadas = [
        {chave: str(valor) for chave, valor in linha.items()}
        for linha in linhas
    ]

    assert recuperadas == esperadas, f"Falha na gravação de {nome}"
    print(f"{nome}: {len(recuperadas)} registros, conferido")

trechos_v2.csv: 279 registros, conferido
configuracao_trechos_v2.csv: 1 registros, conferido


In [15]:
textos_trechos_v2 = [
    trecho["texto_trecho"]
    for trecho in trechos_corpus_v2
]

embeddings_trechos_v2 = modelo_embeddings.encode(
    textos_trechos_v2,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

assert embeddings_trechos_v2.shape == (
    len(trechos_corpus_v2),
    modelo_embeddings.get_embedding_dimension(),
)
assert np.isfinite(embeddings_trechos_v2).all()
assert np.allclose(
    np.linalg.norm(embeddings_trechos_v2, axis=1),
    1.0,
    atol=1e-5,
)

print("Formato:", embeddings_trechos_v2.shape)
print("Valores finitos e normalização: conferidos")

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Formato: (279, 768)
Valores finitos e normalização: conferidos


In [16]:
pergunta_busca_v2 = "Como solicitar o trancamento de matrícula?"

vetor_pergunta_v2 = modelo_embeddings.encode(
    pergunta_busca_v2,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

similaridades_v2 = embeddings_trechos_v2 @ vetor_pergunta_v2
indices_v2 = np.argsort(-similaridades_v2)[:3]

for ordem, indice in enumerate(indices_v2, start=1):
    trecho = trechos_corpus_v2[int(indice)]

    print(
        f"\n{ordem}. {trecho['arquivo']} | "
        f"página: {trecho.get('pagina') or 'Não se aplica'} | "
        f"similaridade: {similaridades_v2[indice]:.4f}"
    )
    print("ID:", trecho["id_trecho"])
    print(trecho["texto_trecho"])


1. GradManualAlunoIbmec.pdf | página: 38 | similaridade: 0.7937
ID: r028_t001
Trancamento de matrícula
Até a data limite constante no calendário acadêmico o aluno poderá solicitar
o trancamento da matrícula. O atendimento da solicitação de trancamento da
matrícula depende do preenchimento cumulativo das seguintes condições:
(a) a solicitação de entrevista de trancamento/cancelamento deve ser realizada
no Campus Virtual (SIA), no menu “Atendimento > Requerimentos > Novo”,
selecionar a categoria “Matrícula/Transferência > Entrevista Trancamento/
Cancelamento de Matrícula”

2. GradManualAlunoIbmec.pdf | página: 38 | similaridade: 0.7878
ID: r028_t002
/Transferência > Entrevista Trancamento/
Cancelamento de Matrícula”, após a unidade entrará em contato para realizar a
entrevista e abrir o requerimento de trancamento de matrícula;
(b) a matrícula deve estar efetiva e vigente ao tempo da solicitação de trancamento;
(c) o aluno deve atender todas as regras relativas ao trancamento da matrícu

In [17]:
import hashlib

caminho_vetores = pasta_saida / "embeddings_trechos_v2.npy"
caminho_indice = pasta_saida / "indice_embeddings_v2.csv"

for caminho in [caminho_vetores, caminho_indice]:
    if caminho.exists():
        raise FileExistsError(f"{caminho.name} já existe.")

indice_embeddings_v2 = [
    {
        "linha": numero,
        "id_trecho": trecho["id_trecho"],
        "sha256_texto": hashlib.sha256(
            trecho["texto_trecho"].encode("utf-8")
        ).hexdigest(),
        "modelo": nome_modelo,
    }
    for numero, trecho in enumerate(trechos_corpus_v2)
]

with caminho_vetores.open("xb") as arquivo:
    np.save(arquivo, embeddings_trechos_v2, allow_pickle=False)

with caminho_indice.open(
    "x", encoding="utf-8-sig", newline=""
) as arquivo:
    escritor = csv.DictWriter(
        arquivo,
        fieldnames=list(indice_embeddings_v2[0]),
        delimiter=";",
    )
    escritor.writeheader()
    escritor.writerows(indice_embeddings_v2)

vetores_recuperados = np.load(
    caminho_vetores, allow_pickle=False
)
assert np.array_equal(
    vetores_recuperados, embeddings_trechos_v2
)

with caminho_indice.open(
    encoding="utf-8-sig", newline=""
) as arquivo:
    indice_recuperado = list(
        csv.DictReader(arquivo, delimiter=";")
    )

assert indice_recuperado == [
    {chave: str(valor) for chave, valor in linha.items()}
    for linha in indice_embeddings_v2
]

print("Embeddings salvos e conferidos:", vetores_recuperados.shape)
print("Índice salvo e conferido:", len(indice_recuperado))

Embeddings salvos e conferidos: (279, 768)
Índice salvo e conferido: 279


In [18]:
perguntas_avaliacao = [
    {
        "id_pergunta": "p01",
        "pergunta": "Como enviar a documentação de estágio pelo SIA?",
    },
    {
        "id_pergunta": "p02",
        "pergunta": "Como acessar os livros da Minha Biblioteca?",
    },
    {
        "id_pergunta": "p03",
        "pergunta": "Como solicitar a carteirinha de estudante?",
    },
    {
        "id_pergunta": "p04",
        "pergunta": "Como é calculada a média final das disciplinas?",
    },
    {
        "id_pergunta": "p05",
        "pergunta": "O que são atividades acadêmicas complementares?",
    },
]

for item in perguntas_avaliacao:
    print(f'{item["id_pergunta"]}: {item["pergunta"]}')

p01: Como enviar a documentação de estágio pelo SIA?
p02: Como acessar os livros da Minha Biblioteca?
p03: Como solicitar a carteirinha de estudante?
p04: Como é calculada a média final das disciplinas?
p05: O que são atividades acadêmicas complementares?


In [19]:
from IPython.display import display, HTML
from html import escape

candidatos_p01 = [
    trecho for trecho in trechos_corpus_v2
    if trecho["arquivo"] == "07.html"
]

blocos = []

for trecho in candidatos_p01:
    blocos.append(
        f"<h4>{escape(trecho['id_trecho'])}</h4>"
        f"<p>{escape(trecho['texto_trecho'])}</p>"
    )

print("Pergunta:", perguntas_avaliacao[0]["pergunta"])
print("Trechos da fonte:", len(candidatos_p01))

display(HTML(
    '<div style="max-height: 450px; overflow-y: auto; '
    'white-space: pre-wrap;">'
    + "\n".join(blocos)
    + "</div>"
))

Pergunta: Como enviar a documentação de estágio pelo SIA?
Trechos da fonte: 5


In [20]:
referencias_recuperacao = {
    "p01": {"r086_t003"},
}

In [21]:
candidatos_p02 = [
    trecho for trecho in trechos_corpus_v2
    if trecho["arquivo"] in {"09.html", "10.html"}
]

blocos = [
    f"<h4>{escape(t['id_trecho'])} | {escape(t['arquivo'])}</h4>"
    f"<p>{escape(t['texto_trecho'])}</p>"
    for t in candidatos_p02
]

print("Pergunta:", perguntas_avaliacao[1]["pergunta"])

display(HTML(
    '<div style="max-height: 450px; overflow-y: auto; '
    'white-space: pre-wrap;">'
    + "\n".join(blocos)
    + "</div>"
))

Pergunta: Como acessar os livros da Minha Biblioteca?


In [25]:
import re

quantidade_10 = sum(
    trecho["arquivo"] == "10.html"
    for trecho in trechos_corpus_v2
)

candidatos_p02_corpus = [
    trecho for trecho in trechos_corpus_v2
    if re.search(
        r"minha\s+biblioteca",
        trecho["texto_trecho"],
        flags=re.IGNORECASE,
    )
]

print("Trechos de 10.html:", quantidade_10)
print("Trechos que mencionam Minha Biblioteca:",
      len(candidatos_p02_corpus))

for trecho in candidatos_p02_corpus:
    print(
        trecho["id_trecho"],
        "|", trecho["arquivo"],
        "| página:", trecho.get("pagina") or "Não se aplica",
    )

Trechos de 10.html: 0
Trechos que mencionam Minha Biblioteca: 3
r065_t001 | GuiaCalouroIbmecFL.pdf | página: 32
r088_t001 | 09.html | página: Não se aplica
r089_t001 | 17.html | página: Não se aplica


In [26]:
ids_para_conferir = {"r065_t001", "r089_t001"}

blocos = [
    f"<h4>{escape(t['id_trecho'])} | {escape(t['arquivo'])}</h4>"
    f"<p>{escape(t['texto_trecho'])}</p>"
    for t in trechos_corpus_v2
    if t["id_trecho"] in ids_para_conferir
]

display(HTML(
    '<div style="max-height: 450px; overflow-y: auto; '
    'white-space: pre-wrap;">'
    + "\n".join(blocos)
    + "</div>"
))

In [27]:
referencias_recuperacao["p02"] = {"r065_t001"}

In [28]:
candidatos_p03 = [
    trecho for trecho in trechos_corpus_v2
    if re.search(
        r"carteirinh|carteira\s+(?:de\s+estudante|estudantil)",
        trecho["texto_trecho"],
        flags=re.IGNORECASE,
    )
]

print("Trechos candidatos:", len(candidatos_p03))

blocos = [
    f"<h4>{escape(t['id_trecho'])} | "
    f"{escape(t['arquivo'])} | página: "
    f"{escape(str(t.get('pagina') or 'Não se aplica'))}</h4>"
    f"<p>{escape(t['texto_trecho'])}</p>"
    for t in candidatos_p03
]

display(HTML(
    '<div style="max-height: 450px; overflow-y: auto; '
    'white-space: pre-wrap;">'
    + "\n".join(blocos)
    + "</div>"
))

Trechos candidatos: 5


In [29]:
referencias_recuperacao["p03"] = {
    "r085_t007",
    "r085_t008",
}

In [30]:
candidatos_p04 = [
    trecho for trecho in trechos_corpus_v2
    if re.search(
        r"m[eé]dia\s+final|\bMF\s*=",
        trecho["texto_trecho"],
        flags=re.IGNORECASE,
    )
]

print("Trechos candidatos:", len(candidatos_p04))

blocos = [
    f"<h4>{escape(t['id_trecho'])} | "
    f"{escape(t['arquivo'])} | página: "
    f"{escape(str(t.get('pagina') or 'Não se aplica'))}</h4>"
    f"<p>{escape(t['texto_trecho'])}</p>"
    for t in candidatos_p04
]

display(HTML(
    '<div style="max-height: 450px; overflow-y: auto; '
    'white-space: pre-wrap;">'
    + "\n".join(blocos)
    + "</div>"
))

Trechos candidatos: 0


In [31]:
import unicodedata

def normalizar_para_busca(texto):
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(
        caractere for caractere in texto
        if not unicodedata.combining(caractere)
    )
    return re.sub(r"\s+", " ", texto).lower()


candidatos_p04 = [
    trecho for trecho in trechos_corpus_v2
    if re.search(
        r"\bmedia\b|\bap\s*[12]\b|avaliacao continuada|0\s*[,.]\s*4",
        normalizar_para_busca(trecho["texto_trecho"]),
    )
]

print("Trechos candidatos:", len(candidatos_p04))

blocos = [
    f"<h4>{escape(t['id_trecho'])} | "
    f"{escape(t['arquivo'])} | página: "
    f"{escape(str(t.get('pagina') or 'Não se aplica'))}</h4>"
    f"<p>{escape(t['texto_trecho'])}</p>"
    for t in candidatos_p04
]

display(HTML(
    '<div style="max-height: 450px; overflow-y: auto; '
    'white-space: pre-wrap;">'
    + "\n".join(blocos)
    + "</div>"
))

Trechos candidatos: 9


In [32]:
caminho_exclusoes = (
    raiz / "data" / "processed" / "exclusoes_corpus_v3.csv"
)

with caminho_exclusoes.open(
    encoding="utf-8-sig", newline=""
) as arquivo:
    exclusoes = list(csv.DictReader(arquivo, delimiter=";"))

exclusoes_avaliacoes = [
    registro for registro in exclusoes
    if re.search(
        r"avaliacao|substitutiva|divergencia|media",
        normalizar_para_busca(
            " ".join(str(valor or "") for valor in registro.values())
        ),
    )
]

print("Exclusões relacionadas:", len(exclusoes_avaliacoes))

for registro in exclusoes_avaliacoes:
    print()
    for campo, valor in registro.items():
        if valor:
            print(f"{campo}: {valor}")

Exclusões relacionadas: 2

arquivo: GradManualAlunoIbmec.pdf
pagina: 29
motivo: Divergência documental sobre elegibilidade para AS com média 7: manual informa igual ou superior a 7; guia informa maior que 7. Vigência e precedência ainda não confirmadas.

arquivo: GuiadoAlunoIbmecFariaLima.pdf
pagina: 4
motivo: Divergência documental sobre elegibilidade para AS com média 7: manual informa igual ou superior a 7; guia informa maior que 7. Vigência e precedência ainda não confirmadas.


In [33]:
referencias_recuperacao["p04"] = set()

observacoes_recuperacao = {
    "p04": (
        "Fórmula não localizada no corpus v3. As páginas de avaliações "
        "foram excluídas integralmente por divergência sobre AS. "
        "A divergência registrada não é sobre a fórmula da média."
    ),
}

In [34]:
candidatos_p05 = [
    trecho for trecho in trechos_corpus_v2
    if re.search(
        r"\baac\b|atividades?\s+(academicas?\s+)?complementares?",
        normalizar_para_busca(trecho["texto_trecho"]),
    )
]

print("Trechos candidatos:", len(candidatos_p05))

blocos = [
    f"<h4>{escape(t['id_trecho'])} | "
    f"{escape(t['arquivo'])} | página: "
    f"{escape(str(t.get('pagina') or 'Não se aplica'))}</h4>"
    f"<p>{escape(t['texto_trecho'])}</p>"
    for t in candidatos_p05
]

display(HTML(
    '<div style="max-height: 450px; overflow-y: auto; '
    'white-space: pre-wrap;">'
    + "\n".join(blocos)
    + "</div>"
))

Trechos candidatos: 13


In [35]:
referencias_recuperacao["p05"] = {
    "r021_t002",
    "r061_t001",
}

ids_existentes = {
    trecho["id_trecho"] for trecho in trechos_corpus_v2
}

for item in perguntas_avaliacao:
    id_pergunta = item["id_pergunta"]
    referencias = referencias_recuperacao[id_pergunta]

    assert referencias <= ids_existentes

    print(
        id_pergunta,
        "| referências:", sorted(referencias),
        "|", "avaliar recuperação" if referencias
        else "sem referência suficiente identificada",
    )

p01 | referências: ['r086_t003'] | avaliar recuperação
p02 | referências: ['r065_t001'] | avaliar recuperação
p03 | referências: ['r085_t007', 'r085_t008'] | avaliar recuperação
p04 | referências: [] | sem referência suficiente identificada
p05 | referências: ['r021_t002', 'r061_t001'] | avaliar recuperação


In [36]:
resultados_recuperacao = []

for item in perguntas_avaliacao:
    id_pergunta = item["id_pergunta"]
    referencias = referencias_recuperacao[id_pergunta]

    vetor = modelo_embeddings.encode(
        item["pergunta"],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    pontuacoes = embeddings_trechos_v2 @ vetor
    indices = np.argsort(-pontuacoes)[:3]

    ids_recuperados = [
        trechos_corpus_v2[int(indice)]["id_trecho"]
        for indice in indices
    ]

    encontradas = referencias & set(ids_recuperados)
    recall = (
        len(encontradas) / len(referencias)
        if referencias else None
    )

    resultados_recuperacao.append({
        "id_pergunta": id_pergunta,
        "pergunta": item["pergunta"],
        "referencias": sorted(referencias),
        "recuperados": ids_recuperados,
        "similaridades": [
            float(pontuacoes[indice]) for indice in indices
        ],
        "recall_3": recall,
    })

    print(f"\n{id_pergunta}: {item['pergunta']}")
    print("Top 3:", ids_recuperados)
    print("Referências encontradas:", sorted(encontradas))
    print(
        "Recall@3:",
        f"{recall:.2%}" if recall is not None
        else "Não calculado",
    )

recalls_validos = [
    resultado["recall_3"]
    for resultado in resultados_recuperacao
    if resultado["recall_3"] is not None
]

print(
    "\nRecall@3 médio sobre as referências anotadas:",
    f"{np.mean(recalls_validos):.2%}",
)


p01: Como enviar a documentação de estágio pelo SIA?
Top 3: ['r086_t003', 'r086_t004', 'r086_t001']
Referências encontradas: ['r086_t003']
Recall@3: 100.00%

p02: Como acessar os livros da Minha Biblioteca?
Top 3: ['r065_t001', 'r089_t001', 'r016_t002']
Referências encontradas: ['r065_t001']
Recall@3: 100.00%

p03: Como solicitar a carteirinha de estudante?
Top 3: ['r085_t007', 'r085_t008', 'r029_t002']
Referências encontradas: ['r085_t007', 'r085_t008']
Recall@3: 100.00%

p04: Como é calculada a média final das disciplinas?
Top 3: ['r022_t006', 'r022_t005', 'r022_t004']
Referências encontradas: []
Recall@3: Não calculado

p05: O que são atividades acadêmicas complementares?
Top 3: ['r061_t001', 'r021_t002', 'r062_t001']
Referências encontradas: ['r021_t002', 'r061_t001']
Recall@3: 100.00%

Recall@3 médio sobre as referências anotadas: 100.00%


In [37]:
import json

pasta_avaliacao = raiz / "reports" / "recuperacao_v1"
pasta_avaliacao.mkdir(parents=True, exist_ok=False)

linhas_avaliacao = [
    {
        "id_pergunta": resultado["id_pergunta"],
        "pergunta": resultado["pergunta"],
        "referencias": json.dumps(
            resultado["referencias"], ensure_ascii=False
        ),
        "recuperados": json.dumps(resultado["recuperados"]),
        "similaridades": json.dumps(resultado["similaridades"]),
        "recall_3": (
            resultado["recall_3"]
            if resultado["recall_3"] is not None else ""
        ),
        "observacao": observacoes_recuperacao.get(
            resultado["id_pergunta"], ""
        ),
    }
    for resultado in resultados_recuperacao
]

caminho_avaliacao = pasta_avaliacao / "avaliacao.csv"

with caminho_avaliacao.open(
    "x", encoding="utf-8-sig", newline=""
) as arquivo:
    escritor = csv.DictWriter(
        arquivo,
        fieldnames=list(linhas_avaliacao[0]),
        delimiter=";",
    )
    escritor.writeheader()
    escritor.writerows(linhas_avaliacao)

with caminho_avaliacao.open(
    encoding="utf-8-sig", newline=""
) as arquivo:
    linhas_recuperadas = list(
        csv.DictReader(arquivo, delimiter=";")
    )

assert linhas_recuperadas == [
    {chave: str(valor) for chave, valor in linha.items()}
    for linha in linhas_avaliacao
]

print("Avaliação salva e conferida:", len(linhas_recuperadas), "perguntas")

Avaliação salva e conferida: 5 perguntas


In [38]:
from importlib.metadata import version

configuracao_recuperacao = [{
    "modelo": nome_modelo,
    "revisao_modelo": (
        getattr(
            modelo_embeddings[0].auto_model.config,
            "_commit_hash",
            None,
        ) or "Não identificada"
    ),
    "sentence_transformers": version("sentence-transformers"),
    "transformers": version("transformers"),
    "numpy": np.__version__,
    "arquivo_trechos": "data/processed/trechos_v2.csv",
    "arquivo_embeddings": "data/processed/embeddings_trechos_v2.npy",
    "arquivo_indice": "data/processed/indice_embeddings_v2.csv",
    "sha256_trechos": hashlib.sha256(
        (pasta_saida / "trechos_v2.csv").read_bytes()
    ).hexdigest(),
    "sha256_embeddings": hashlib.sha256(
        caminho_vetores.read_bytes()
    ).hexdigest(),
    "quantidade_trechos": len(trechos_corpus_v2),
    "dimensoes": embeddings_trechos_v2.shape[1],
    "embeddings_normalizados": True,
    "similaridade": "Cosseno por produto escalar de vetores normalizados",
    "k": 3,
    "quantidade_perguntas": len(resultados_recuperacao),
    "perguntas_com_referencia": len(recalls_validos),
    "recall_3_medio": float(np.mean(recalls_validos)),
    "tipo_avaliacao": "Piloto com perguntas sintéticas",
    "limite_gabarito": "Referências anotadas não exaustivas",
    "pergunta_sem_referencia": "p04; excluída do cálculo de Recall",
}]

caminho_configuracao = pasta_avaliacao / "configuracao.csv"

with caminho_configuracao.open(
    "x", encoding="utf-8-sig", newline=""
) as arquivo:
    escritor = csv.DictWriter(
        arquivo,
        fieldnames=list(configuracao_recuperacao[0]),
        delimiter=";",
    )
    escritor.writeheader()
    escritor.writerows(configuracao_recuperacao)

with caminho_configuracao.open(
    encoding="utf-8-sig", newline=""
) as arquivo:
    configuracao_recuperada = list(
        csv.DictReader(arquivo, delimiter=";")
    )

assert configuracao_recuperada == [
    {chave: str(valor) for chave, valor in linha.items()}
    for linha in configuracao_recuperacao
]

print("Configuração salva e conferida: 1 registro")

Configuração salva e conferida: 1 registro
